# Passo 1: Preparação Simplificada dos Dados (Padding)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Amostra de comentários do e-commerce
comentarios = [
    "O produto é maravilhoso, super recomendo",
    "Entrega rápida e excelente atendimento",
    "Detestei o produto, veio quebrado e estragado",
    "Péssima qualidade, não comprem de jeito nenhum"
]

# 1 = Positivo, 0 = Negativo
labels = np.array([1, 1, 0, 0])

# Criando o Tokenizer e convertendo os textos em números
tokenizer = Tokenizer(num_words=1000, oov_token="<UNK>")
tokenizer.fit_on_texts(comentarios)
sequencias = tokenizer.texts_to_sequences(comentarios)

# Padronizando o tamanho das frases para ter exatamente 6 palavras (Padding)
dados_prontos = pad_sequences(sequencias, maxlen=6, padding='post')

print("Frases transformadas em sequências numéricas fixas:\n", dados_prontos)

# Passo 2: Construindo a Rede Neural Sequencial (Opção A: RNN Clássica)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# [Sequência de IDs Inteiros] -> (1. Embedding) -> (2. SimpleRNN) -> (3. Dense) -> [Sentimento (0 ou 1)]

vocab_size = 1000  # Tamanho máximo do vocabulário
max_len = 6        # Tamanho de cada sequência de entrada

model_rnn = Sequential([
    # Camada 1: Transforma IDs em vetores contínuos (Word Embeddings)
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    
    # Camada 2: A Célula Recorrente Simples (RNN) que processa palavra por palavra
    SimpleRNN(units=8),
    
    # Camada 3: Neurônio de saída (Sigmóide mapeia o resultado entre 0 e 1)
    Dense(units=1, activation='sigmoid')
])

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

# Passo 2: Construindo a Rede Neural Sequencial (Opção B: LSTM)

In [ ]:
from tensorflow.keras.layers import LSTM

model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    
    # Mudança conceitual: Agora o fluxo possui portões de retenção de longo prazo
    LSTM(units=8),
    
    Dense(units=1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.summary()

# Passo 3: Treinamento e Predição

In [ ]:
# Treinando o modelo LSTM com os nossos dados de exemplo
model_lstm.fit(dados_prontos, labels, epochs=5, verbose=0)
model_rnn.fit(dados_prontos, labels, epochs=5, verbose=0)

# Testando uma frase nova
nova_frase = ["O atendimento foi excelente"]
nova_seq = tokenizer.texts_to_sequences(nova_frase)
nova_seq_padded = pad_sequences(nova_seq, maxlen=6, padding='post')

predicao = model_rnn.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

predicao = model_lstm.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

# Transformers - Passo 1: Instanciação Simples de um Pipeline

In [ ]:
# Instale a biblioteca necessária
# !pip install transformers

from transformers import pipeline

# Criando um classificador de análise de sentimento usando um Transformer padrão (BERT)
classificador = pipeline("sentiment-analysis")

# Testando uma frase que confunde os modelos clássicos (como negações ou sarcasmo)
resultado = classificador("Eu não acho que o produto seja ruim, pelo contrário, superou minhas expectativas.")
print(resultado)

# Transformers - Passo 2: Abrindo a Caixa-Preta

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

nome_modelo = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
modelo = AutoModelForSequenceClassification.from_pretrained(nome_modelo)

frase = "Transformers are amazing at context."

# 1. Veja como o Tokenizer transforma o texto em IDs numéricos e adiciona tokens especiais
inputs = tokenizer(frase, return_tensors="pt")
print("Tokens convertidos em IDs numéricos:\n", inputs["input_ids"])

# 2. Passe os IDs pelo modelo para obter as predições (Logits)
with torch.no_grad():
    outputs = modelo(**inputs)

print("\nSaída bruta matemática do modelo (Logits):\n", outputs.logits)